In [22]:
!pip install "numpy>=2.0" pandas requests statsmodels great_tables adjustText fastparquet tabulate pyarrow sportsdataverse
!pip install nfl_data_py --no-deps
!pip install appdirs

import os
import tabulate
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nfl_data_py as nfl
import statsmodels.formula.api as smf
import pyarrow
import sportsdataverse as sdv
import polars as pl
from great_tables import GT, md
from adjustText import adjust_text

  Using cached numpy-2.5.2-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.5.2-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nfl-data-py 0.3.3 requires numpy<2.0,>=1.0, but you have numpy 2.5.2 which is incompatible.
nfl-data-py 0.3.3 requires pandas<2.0,>=1.0, but you have pandas 3.0.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.2 which is incompatible.


In [23]:
# ---------------------------------------------------------
# 1. Load play-by-play data (2021–2025)
# ---------------------------------------------------------
seasons = [2021, 2022, 2023, 2024, 2025]

pbp = sdv.cfb.load_cfb_pbp(seasons=seasons, return_as_pandas=False)
schedule = sdv.cfb.load_cfb_schedule(seasons=seasons, return_as_pandas=False)

print("PBP rows:", pbp.shape)
print("Schedule rows:", schedule.shape)

PBP rows: (778447, 477)
Schedule rows: (17431, 31)


In [24]:
conference_cols = [c for c in schedule.columns if "conference" in c.lower()]
conference_cols

['conference_game', 'home_conference', 'away_conference']

In [25]:
# ---------------------------------------------------------
# 2. Restrict to P4 conferences using schedule metadata
# ---------------------------------------------------------
schedule = sdv.cfb.load_cfb_schedule(
    seasons=seasons,
    return_as_pandas=False
)

p4 = ["SEC", "ACC", "Big Ten", "Big 12"]

schedule_small = schedule.select([
    "game_id", "home_conference", "away_conference"
])

pbp = pbp.join(schedule_small, on="game_id", how="left")

pbp = pbp.filter(
    pl.col("home_conference").is_in(p4) |
    pl.col("away_conference").is_in(p4)
)

In [26]:
print("week" in pbp.columns)
print([c for c in pbp.columns if "week" in c.lower()])

True
['week']


In [30]:
# ---------------------------------------------------------
# 3. Split season into first half vs second half
# ---------------------------------------------------------
pbp = pbp.with_columns([
    pl.when(pl.col("week") <= 6)
      .then(pl.lit("first_half"))
      .otherwise(pl.lit("second_half"))
      .alias("season_half")
])

pbp.select(["week", "season_half"])

week,season_half
i64,str
1,"""first_half"""
1,"""first_half"""
1,"""first_half"""
1,"""first_half"""
1,"""first_half"""
…,…
1,"""first_half"""
1,"""first_half"""
1,"""first_half"""


In [48]:
# ---------------------------------------------------------
# List of metrics to choose from
# ---------------------------------------------------------
offense_cols = [c for c in pbp.columns if "pos_team" in c or "yards" in c or "success" in c or "EPA" in c]
print("Offensive Columns:")
print(offense_cols)

defense_cols = [c for c in pbp.columns if "def_pos_team" in c or "yards" in c or "success" in c or "EPA" in c]
print("Defensive Columns:")
print(defense_cols)

Offensive Columns:
['pos_team_id', 'pos_team', 'def_pos_team_id', 'def_pos_team', 'pos_team_score', 'def_pos_team_score', 'EPA', 'def_EPA', 'change_of_pos_team', 'lead_pos_team', 'lag_pos_team', 'lead_pos_team2', 'lag_change_of_pos_team', 'start.yardsToEndzone', 'end.yardsToEndzone', 'drive.yards', 'lead_start_yardsToEndzone', 'start.pos_team.id', 'start.def_pos_team.id', 'end.def_pos_team.id', 'end.pos_team.id', 'start.pos_team.name', 'start.def_pos_team.name', 'end.pos_team.name', 'end.def_pos_team.name', 'start.pos_team_score', 'start.def_pos_team_score', 'end.pos_team_score', 'end.def_pos_team_score', 'start.pos_team_receives_2H_kickoff', 'end.pos_team_receives_2H_kickoff', 'air_yardsToEndzone', 'air_yards', 'yards_after_catch', 'is_pos_team_turnover', 'is_def_pos_team_turnover', 'penalty_yards_signed', 'line_yards', 'second_level_yards', 'open_field_yards', 'highlight_yards', 'opp_highlight_yards', 'short_rush_success', 'power_rush_success', 'start.pos_team_spread', 'end.pos_team_

In [33]:
# ---------------------------------------------------------
# 4. Offensive metrics to evaluate
# ---------------------------------------------------------
pbp = pbp.with_columns([
    (pl.col("yds_rushed").fill_null(0) +
     pl.col("yds_receiving").fill_null(0) -
     pl.col("yds_sacked").fill_null(0)
    ).alias("yards_gained")
])

# play_success
pbp = pbp.with_columns(
    pl.when(pl.col("down") == 1)
    .then(pl.col("yards_gained") >= (0.4 * pl.col("distance")))
    .when(pl.col("down") == 2)
    .then(pl.col("yards_gained") >= (0.6 * pl.col("distance")))
    .when((pl.col("down") == 3) | (pl.col("down") == 4))
    .then(pl.col("yards_gained") >= pl.col("distance"))
    .otherwise(pl.lit(False))
    .alias("play_success")
)

offensive_metrics = {
    "yards_per_play": pl.col("yards_gained"),
    "success_rate": pl.col("play_success"),
    "epa": pl.col("EPA")
}

team_half_off = (
    pbp.group_by(["pos_team", "season_half"])
       .agg([offensive_metrics[m].mean().alias(m) for m in offensive_metrics])
)

team_pivot_off = team_half_off.pivot(
    values=list(offensive_metrics.keys()),
    index="pos_team",
    columns="season_half"
)

team_pivot_off.head()

/tmp/ipykernel_2796/3178754316.py:35: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  team_pivot_off = team_half_off.pivot(


pos_team,yards_per_play_first_half,yards_per_play_second_half,success_rate_first_half,success_rate_second_half,epa_first_half,epa_second_half
str,f64,f64,f64,f64,f64,f64
"""Nebraska Cornhuskers""",5.217837,4.374014,0.412144,0.359485,0.051723,-0.045958
"""Ohio State Buckeyes""",6.311651,5.542367,0.467682,0.443332,0.242396,0.141602
"""NC State Wolfpack""",4.702205,4.664291,0.388146,0.357482,0.067949,0.045754
"""Tennessee Volunteers""",5.314095,5.370251,0.428413,0.42112,0.164864,0.130555
"""Duke Blue Devils""",5.013453,4.365488,0.393124,0.340479,0.098212,0.019822


In [44]:
off_results = {}

for metric in offensive_metrics.keys():
    first = team_pivot_off[f"{metric}_first_half"]
    second = team_pivot_off[f"{metric}_second_half"]
    corr = pl.DataFrame().select(pl.corr(first, second)).item()
    off_results[metric] = corr

print("=== Offensive Metric Stability ===")
for metric, corr in off_results.items():
    print(f"{metric}: {corr:.3f}")

=== Offensive Metric Stability ===
yards_per_play: 0.753
success_rate: 0.705
epa: 0.610


In [40]:
defensive_metrics = {
    "yards_allowed_per_play": pl.col("yards_gained"),
    "success_allowed": pl.col("play_success"),
    "epa_allowed": pl.col("EPA")
}

team_half_def = (
    pbp.group_by(["def_pos_team", "season_half"])
       .agg([defensive_metrics[m].mean().alias(m) for m in defensive_metrics])
)

team_pivot_def = team_half_def.pivot(
    values=list(defensive_metrics.keys()),
    index="def_pos_team",
    columns="season_half"
)

team_pivot_def.head()

/tmp/ipykernel_2796/3759898311.py:12: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  team_pivot_def = team_half_def.pivot(


def_pos_team,yards_allowed_per_play_first_half,yards_allowed_per_play_second_half,success_allowed_first_half,success_allowed_second_half,epa_allowed_first_half,epa_allowed_second_half
str,f64,f64,f64,f64,f64,f64
"""Nebraska Cornhuskers""",4.198589,4.693836,0.360802,0.397405,-0.034068,0.046359
"""Ohio State Buckeyes""",3.77935,3.958834,0.32696,0.330783,-0.066934,-0.074267
"""NC State Wolfpack""",4.421251,4.685876,0.319894,0.334087,-0.022017,0.036462
"""South Florida Bulls""",5.595899,null,0.379976,null,0.087631,null
"""Tennessee Volunteers""",4.005558,4.539864,0.334198,0.361557,-0.034898,0.032344


In [42]:
def_results = {}

for metric in defensive_metrics.keys():
    first = team_pivot_def[f"{metric}_first_half"]
    second = team_pivot_def[f"{metric}_second_half"]
    corr = pl.DataFrame().select(pl.corr(first, second)).item()
    def_results[metric] = corr

print("=== Defensive Metric Stability ===")
for metric, corr in def_results.items():
    print(f"{metric}: {corr:.3f}")

=== Defensive Metric Stability ===
yards_allowed_per_play: 0.710
success_allowed: 0.669
epa_allowed: 0.624


##Metric Stability Conclusion: Offense AND Defense for the P4 conference

These conclusions were based on correlations between the first and second half of the season during the 2021 to 2025 seasons.

### Offensive Metric Stability:
For offense, I completed analysis for:

Yards per Play     
Success Rate      
EPA      

**Conclusion:**
All three offensive metrics demonstrate meaningful stability from the first half of the season to the second half.
1. Yards per Play shows the strongest correlation (0.753), making it the most reliable offensive predictor across the season. This indicates that teams who generate efficient yardage early tend to maintain that efficiency later.

2. Success Rate exhibits strong stability (0.705). Because success rate measures consistency rather than explosiveness, its high correlation suggests that consistent offenses remain consistent across the season.

3. EPA, while still moderately stable (0.610), is the least predictive of the three offensive metrics. EPA is more sensitive to explosive plays, situational leverage, and opponent strength, which likely explains its lower stability. As demonstrated, situational factors and schedule variability can introduce instability into these evaluations. Rivalry environments and fan engagement often create performance swings that do not appear in more stable efficiency metrics like the previous two metrics above.

I chose these three metrics because many of the other offensive options introduced too much noise. Metrics like total points or drive yards can be misleading. Teams can play extremely well on offense even in lower scoring games, especially if the opposing defense performs strongly. The metrics I selected showed stable correlations and provide a clearer, more consistent picture of offensive performance throughout the season. Yards per play is neccessary for any offense team to win no matter what, a team can't score if they don't go down the field.

### Defensive Metric Stability
For defense I completed in analysis for:

Yards Allowed per Play     
Success Allowed       
EPA Allowed      

**Conclusion:**
Similar to offensive metrics, defensive metrics also demonstrate good stability from the first to the second half of the season.
1. 'Yards Allowed per Play' is the most stable metric (0.710)
2. 'Success Allowed' (0.669)
3. 'EPA Allowed' (0.624)

### Overall Summary

Because the first half and second half metrics show strong correlations, we can see that teams generally carry their performance from early in the season into the later weeks. In other words, teams that perform well in the first half are likely to maintain that momentum in the second half on both offense and defense.Where teams that perform poorly, may lose sight of the end if there is no more playoff chances.